# Tambahan - Klasifikasi
Membandingkan beberapa algoritma pada dataset **Pima Indians Diabetes** (prediksi pasien diabetes atau tidak).
Nilai 0 pada kolom `glucose`, `blood_pressure`, `skin_thickness`, `insulin`, `bmi` tidak valid, jadi dianggap missing lalu diimputasi (median).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [2]:
df = pd.read_csv('../dataset/diabetes.csv')
kolom_nol = ['glucose','blood_pressure','skin_thickness','insulin','bmi']
df[kolom_nol] = df[kolom_nol].replace(0, np.nan)
print('Missing per kolom:'); print(df[kolom_nol].isna().sum())
X, y = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape

Missing per kolom:
glucose             5
blood_pressure     35
skin_thickness    227
insulin           374
bmi                11
dtype: int64


((614, 8), (154, 8))

In [3]:
imp = lambda: SimpleImputer(strategy='median')
models = {
    'KNN': make_pipeline(imp(), StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'Decision Tree': make_pipeline(imp(), DecisionTreeClassifier(max_depth=5, random_state=42)),
    'Random Forest': make_pipeline(imp(), RandomForestClassifier(n_estimators=200, random_state=42)),
    'Logistic Regression': make_pipeline(imp(), StandardScaler(), LogisticRegression(max_iter=1000)),
}
hasil = {}
for nama, m in models.items():
    cv = cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy')
    m.fit(X_train, y_train)
    hasil[nama] = {'CV mean': cv.mean(), 'CV std': cv.std(), 'Test acc': m.score(X_test, y_test)}
pd.DataFrame(hasil).T.round(4)

,CV mean,CV std,Test acc
KNN,0.7492,0.0249,0.7532
Decision Tree,0.7248,0.0199,0.7597
Random Forest,0.7688,0.0348,0.7403
Logistic Regression,0.7818,0.0125,0.7078


In [4]:
terbaik = max(hasil, key=lambda k: hasil[k]['Test acc'])
print('Model terbaik:', terbaik)
pred = models[terbaik].predict(X_test)
print(classification_report(y_test, pred, target_names=['tidak_diabetes','diabetes']))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.show()

Model terbaik: Decision Tree
                precision    recall  f1-score   support

tidak_diabetes       0.84      0.78      0.81       100
      diabetes       0.64      0.72      0.68        54

      accuracy                           0.76       154
     macro avg       0.74      0.75      0.74       154
  weighted avg       0.77      0.76      0.76       154



## Kesimpulan
- (bandingkan akurasi, precision/recall, kenapa model tertentu lebih baik)